# Cosmos-Transfer2.5 — Video Generation from a Prompt + Input Video

This notebook drives [`examples/inference.py`](inference.py) to run **Cosmos-Transfer2.5-2B** inference. You provide:

1. A **text prompt** describing the world you want.
2. An **input video** (`video_path`).

The model uses a control modality (edge / depth / segmentation / blur) derived from the input video to guide generation. If you don't provide a precomputed control video, it is **computed on the fly** from the input video.

### One-time setup (in a terminal, from the repo root)

This repo pins **Python 3.13** (see `.python-version`). Use the **CUDA 13.0** dependency set (`--extra=cu130`); the CUDA 12.8 build (`cu128`) requires Python 3.10 instead. See the [Setup guide](../docs/setup.md) for system requirements (NVIDIA driver compatible with CUDA 13.0).

```bash
cd /path/to/cosmos-transfer2.5-example

# Install Git LFS once if needed, then fetch real asset files
# (the bundled input video is an LFS pointer until you pull)
sudo apt install -y git-lfs   # skip if already installed
git lfs install
git lfs pull

# Create the environment (CUDA 13.0 build, Python 3.13)
uv python install
uv sync --extra=cu130
source .venv/bin/activate

# Hugging Face auth so checkpoints can download on first run
uv run hf auth login
# then accept the license at https://huggingface.co/nvidia/Cosmos-Transfer2.5-2B
```

### Launch the notebook

Pick whichever fits how you're working:

**A. JupyterLab in the browser**

```bash
uv run --with jupyter jupyter lab examples/transfer2_5_inference.ipynb
```

In Jupyter, select the **`.venv` kernel** (or the kernel that points at this repo's virtual environment).

### Before you run

- One-time setup above is complete and this notebook is running in that environment.
- Example assets are real files, not LFS pointers (`git lfs pull`).
- Hugging Face auth is done and the [NVIDIA Open Model License](https://huggingface.co/nvidia/Cosmos-Transfer2.5-2B) is accepted. Checkpoints download automatically on first inference.
- A single GPU with **~65 GB VRAM** (e.g. H100 / B200) for the 2B model.

See [docs/inference.md](../docs/inference.md) for full details.

## 1. Environment check

Confirm we're at the repo root and that a GPU is visible.

In [1]:
import os
from pathlib import Path

# This notebook lives in examples/ ; run everything from the repo root.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "examples":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)
assert (REPO_ROOT / "examples" / "inference.py").exists(), "Could not find examples/inference.py — run from the repo."

# Show GPU(s)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Repo root: /home/ubuntu/projects/cosmos-transfer2.5-example
name, memory.total [MiB]
NVIDIA L40S, 46068 MiB


## 2. Configure your inputs

Edit the values below. Defaults point at the bundled robot example so you can verify the pipeline end-to-end first, then swap in your own video and prompt.

**Control modality** — pick one:
- `edge` (default): Canny-edge guidance, preserves structure/outlines.
- `depth`: depth-map guidance, preserves 3D scene geometry.
- `seg`: segmentation guidance, preserves object layout.
- `vis`: visual-blur guidance, preserves coarse color/layout.

Leave `control_path` empty (`None`) to compute the control map on the fly from `video_path`.

In [2]:
# ---- Edit these ----
PROMPT = (
    "A sleek humanoid robot arm operating in a bright, modern industrial kitchen "
    "with stainless steel surfaces and warm sunlight streaming through large windows."
)

# Path to YOUR input video. Default = bundled example.
VIDEO_PATH = "assets/robot_example/robot_input.mp4"

# Control modality: one of "edge", "depth", "seg", "vis"
CONTROL = "edge"

# Optional precomputed control video. None => computed on the fly from VIDEO_PATH.
CONTROL_PATH = None

# How strongly the control guides generation (0.0 - 1.0 typical).
CONTROL_WEIGHT = 1.0

# Sampling settings
GUIDANCE = 3       # 0-7: higher = closer adherence to the prompt
NUM_STEPS = 35     # diffusion steps (base model). Use 4 for the distilled model.
SEED = 2025

# Optional: limit frames read from the input video (None = all frames)
NUM_INPUT_FRAMES = None

# Where to write the generated video
OUTPUT_DIR = "outputs/notebook_run"

# Negative prompt (what you DON'T want). None => use the model default.
NEGATIVE_PROMPT = None
# --------------------

assert CONTROL in {"edge", "depth", "seg", "vis"}, CONTROL
assert Path(VIDEO_PATH).exists(), f"Input video not found: {VIDEO_PATH}"
print("Input video:", VIDEO_PATH)
print("Control    :", CONTROL, "(on-the-fly)" if CONTROL_PATH is None else f"({CONTROL_PATH})")
print("Prompt     :", PROMPT)

Input video: assets/robot_example/robot_input.mp4
Control    : edge (on-the-fly)
Prompt     : A sleek humanoid robot arm operating in a bright, modern industrial kitchen with stainless steel surfaces and warm sunlight streaming through large windows.


### Preview the input video

In [3]:
from IPython.display import Video, display

display(Video(VIDEO_PATH, embed=True, width=480))

## 3. Build the inference parameter (spec) JSON

`inference.py` reads a JSON "spec" describing the prompt, input video, and per-modality control settings. We generate it here from the values above.

In [ ]:
import json

# inference.py resolves input paths RELATIVE TO THE SPEC FILE'S DIRECTORY
# (it does os.chdir(spec_path.parent) before validating). Since the spec is
# written into OUTPUT_DIR but our paths are relative to the repo root, we
# convert them to absolute paths so they resolve correctly.
video_path_abs = str(Path(VIDEO_PATH).resolve())

control_block = {"control_weight": CONTROL_WEIGHT}
if CONTROL_PATH is not None:
    control_block["control_path"] = str(Path(CONTROL_PATH).resolve())

spec = {
    "name": "notebook_run",
    "prompt": PROMPT,
    "video_path": video_path_abs,
    "guidance": GUIDANCE,
    "num_steps": NUM_STEPS,
    "seed": SEED,
    CONTROL: control_block,
}
if NUM_INPUT_FRAMES is not None:
    spec["num_input_frames"] = NUM_INPUT_FRAMES
if NEGATIVE_PROMPT is not None:
    spec["negative_prompt"] = NEGATIVE_PROMPT

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
spec_path = Path(OUTPUT_DIR) / "spec.json"
spec_path.write_text(json.dumps(spec, indent=2))
print("Wrote spec to:", spec_path)
print(json.dumps(spec, indent=2))

## 4. Run inference

We invoke `examples/inference.py` as a subprocess (it uses `tyro` for CLI parsing and initializes a distributed environment, so running it as a script is the cleanest approach). The `control:<modality>` subcommand selects the control type.

> **First run downloads the checkpoints (~tens of GB)** and can take a while. Generation itself is roughly a few minutes on an H100/B200 for a 93-frame 720p clip — see the timing tables in [docs/inference.md](../docs/inference.md).

Output is streamed live below.

In [5]:
import subprocess, sys

cmd = [
    sys.executable, "examples/inference.py",
    "-i", str(spec_path),
    "-o", OUTPUT_DIR,
    f"control:{CONTROL}",
]
print("Running:", " ".join(cmd), "\n")

# For multi-GPU, replace the line below with:
#   cmd = ["torchrun", "--nproc_per_node=8", "--master_port=12341", "examples/inference.py",
#          "-i", str(spec_path), "-o", OUTPUT_DIR, f"control:{CONTROL}"]

proc = subprocess.run(cmd)
print("\nExit code:", proc.returncode)
assert proc.returncode == 0, "Inference failed — see the log output above."

Running: /home/ubuntu/.cache/uv/builds-v0/.tmpj3mCoE/bin/python examples/inference.py -i outputs/notebook_run/spec.json -o outputs/notebook_run control:edge 



Error validating parameters from 'outputs/notebook_run/spec.json' at line 0
1 validation error for InferenceArguments
video_path
  Path does not point to a file [type=path_not_file, input_value='assets/robot_example/robot_input.mp4', input_type=str]



Exit code: 1


AssertionError: Inference failed — see the log output above.

## 5. View the generated video

In [ ]:
from IPython.display import Video, display

# Collect mp4s written to the output dir (newest first), excluding our input copies if any.
outputs = sorted(Path(OUTPUT_DIR).rglob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
print("Generated files:")
for p in outputs:
    print("  -", p)

if outputs:
    display(Video(str(outputs[0]), embed=True, width=640))
else:
    print("No .mp4 found in", OUTPUT_DIR)

## Notes & next steps

- **Use the distilled (fast) model**: set `NUM_STEPS = 4` above and add `"--model=edge/distilled"` to `cmd` (distilled is edge-only and expects short, 93-frame clips).
- **Multi-control**: add more than one of `edge`/`depth`/`seg`/`vis` blocks to the `spec`. When mixing modalities the multicontrol model is used automatically; on the CLI you still pass a single `control:` subcommand.
- **Spatiotemporal masks**: add `"mask_path"` (binary mp4, white = apply control) or `"mask_prompt"` (text → SAM2) inside a control block to restrict where a control applies.
- **All parameters**: `python examples/inference.py --help` and `python examples/inference.py control:edge --help`.